# 02 — MLP LOLO Evaluation
# Giai đoạn 2 — Mục 2.1, 2.2, 2.4 — Huấn luyện và đánh giá MLP với LOLO
**Đầu ra**:
 - `outputs/tables/mlp_lolo_results.csv`
 - Các model `.h5` và `.tflite` trong `outputs/models/`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from common import models, quantization, training

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Feature

In [4]:
feature_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/features_mlp.parquet")
feature_cols = [col for col in feature_df.columns if col.startswith(('time_', 'order_', 'envelope_'))]
print(f"Số đặc trưng MLP: {len(feature_cols)}")

Số đặc trưng MLP: 32


# MLP - LOLO

In [5]:
mlp_results = []

for fold_info, train_df, val_df, test_df in training.iterate_lolo_splits(feature_df, load_col='load_hp'):
    print(f"\n--- Fold: {fold_info['fold_name']} ---")
    
    # Chuẩn hóa
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols])
    X_val = scaler.transform(val_df[feature_cols])
    X_test = scaler.transform(test_df[feature_cols])
    
    y_train = train_df['label'].values
    y_val = val_df['label'].values
    y_test = test_df['label'].values
    
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)
    y_test_enc = le.transform(y_test)
    
    # Huấn luyện
    model = models.build_mlp(input_dim=len(feature_cols))
    model = models.compile_classifier(model)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(X_train, y_train_enc,
                        validation_data=(X_val, y_val_enc),
                        epochs=100, batch_size=32,
                        callbacks=[early_stop], verbose=0)
    
    # Đánh giá float
    loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    f1 = f1_score(y_test_enc, y_pred, average='macro')
    
    # Lượng tử hóa INT8
    tflite_bytes = quantization.quantize_model_int8(model, X_train)
    int8_result = quantization.evaluate_tflite_model(tflite_bytes, X_test, y_test_enc)
    
    mlp_results.append({
        'fold': fold_info['fold_name'],
        'test_load': fold_info['test_load'],
        'float_accuracy': acc,
        'float_f1': f1,
        'int8_accuracy': int8_result['accuracy'],
        'epochs': len(history.history['loss'])
    })
    
    # Lưu model và scaler
    model.save(MODELS_DIR / f"mlp_{fold_info['fold_name']}.h5")
    with open(MODELS_DIR / f"scaler_mlp_{fold_info['fold_name']}.pkl", 'wb') as f:
        pickle.dump(scaler, f)
    quantization.model_bytes_to_file(tflite_bytes, MODELS_DIR / f"mlp_{fold_info['fold_name']}.tflite")


--- Fold: test_load_0 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp31dryk2r\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp31dryk2r\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp31dryk2r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1930789724496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930789924176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930789923024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930789922448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930789922832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930789922640: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- Fold: test_load_1 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpw60f4ouk\assets


...

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- Fold: test_load_3 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpv_yo9df1\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpv_yo9df1\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpv_yo9df1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1930798093072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930798091152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930798102480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930798089616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930798102096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930798101136: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [6]:
mlp_results_df = pd.DataFrame(mlp_results)
mlp_results_df.to_csv(TABLES_DIR / "mlp_lolo_results.csv", index=False)
mlp_results_df

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs
0,test_load_0,0,0.7,0.691667,0.9,100
1,test_load_1,1,1.0,1.000000,1.0,100
2,test_load_2,2,1.0,1.000000,1.0,100
3,test_load_3,3,0.9,0.914286,0.8,100
